# 🫀 퀘스트 46 · Q7-B — **SVDB 에서 실제 SE 를 잰다** (오염 차단 설계)

| | **MedKOS / `notebooks/quest46_q7b_svdb_se.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0049`(Q7 필요조건 통과) · `ailab-2026-0047`(Q1) · `ailab-2026-0050`(Q8 기각) |
| 규약 | `SCORING_RULES.md` **R8 · R11 · R11-b · R12 · R13 · R14** |

## 왜 이 실험인가

Q7 은 **양성 수**만 셌다 — SVDB 는 GMIN=10 에서 66개 레코드를 준다. 하지만 Q1 이
가르쳐준 게 정확히 이거다: **양성이 있어도 부트 SE 가 터지면 소용없다.**
SE 는 예측이 있어야 잰다. 그래서 여기서 **처음으로 학습이 들어간다.**

Q8 이 닫아준 문도 여기 걸려 있다 — 가중치로는 개체를 못 만든다(ESS 2.0). 남은 길은
**진짜 개체가 많은 코호트**뿐이고, 그게 SVDB 인지 여기서 확정된다.

## 🔒 오염 차단 — 이 노트북의 설계 원칙

이 실험은 **테스트가 오염되면 전부 무의미**하다. 네 겹으로 막는다.

**① 학습 데이터**: MIT-BIH **DS1 만**. SVDB 는 학습·튜닝·조기종료에 **한 번도** 안 쓴다.
`svdb_leak_audit()` 가 학습 **전에** 돌고, 실패하면 예외를 던져 학습을 시작하지 않는다.

**② `GMIN` 선택**: 여기가 진짜 함정이다. GMIN 을 훑어서 **SE 가 통과하는 값을 고르면
그건 테스트로 고른 것**이다 — R12 가 "검증이 선택에 의존하게 짜지 말라" 고 한 바로 그
사고다. 그래서 SVDB 78레코드를 **DEV / TEST 로 절반씩 미리 가른다**.
`GMIN` 은 **DEV 에서만** 고르고, 관문은 **TEST 에서만** 매긴다.
분할은 Q7 주석 카운트(예측과 무관)로 S 부담을 맞춰 **결정론적으로** 정한다.

**③ 스케일러·특징선택**: `RobustScaler` 는 DS1 에만 fit. WST `SelectKBest` 도
**DS1 에서만** fit 한다(`wst_fit="ds1"`). 실험22-A 는 MIT-BIH 전체로 fit 했는데
(§6.5 와 이어붙이려던 타협) 여기서는 그럴 이유가 없다. 대신 이 값들은
**실험22-A·§6.5 와 직접 비교 불가**다.

**④ 임계값 없음**: 주 지표를 **AUROC**(순위 기반)로 둔다. 동작점을 테스트에서
고르는 오염이 원천적으로 생기지 않는다.

### 공개하는 한계 (누수는 아니지만 숨기지 않는다)

- **transductive 요소**: `_medref`(환자별 중앙 파형)와 BN 적응은 평가 데이터의
  **입력**을 쓴다. 라벨은 안 쓴다. 관문은 **raw(BN 미적응)** 로 매기고 BN 은 참고만.
- **코호트 선택 자체**: SVDB 를 고른 것은 Q7 에서 **라벨 분포를 보고** 한 일이다.
  모델과 무관하지만 "코호트를 골랐다" 는 사실은 남는다.
- **대역 교란**: SVDB 는 128Hz 원본을 360Hz 로 업샘플한 것이라 대역폭이 64Hz 로 잘려
  있다(MIT-BIH 는 180Hz). **그래서 SVDB 전이 낙폭을 인용하면 안 된다.**
  Q7-B 가 묻는 건 낙폭이 아니라 '이 코호트가 판정을 지탱하나' 라 이 교란과 무관하다.
  낙폭이 필요하면 **Q7-C**(동일 대역 MIT-BIH 대조군)를 따로 돌린다.
- **RR 오염 회피**: `svdb_prep.svdb_data.npz` 는 `'+'`(리듬변경) 주석까지 diff 해서
  RR 을 만든다 — **오염됐다**. S 는 RR 로 정의되는 클래스라 치명적이다. 그래서
  `svdb_labels.build_labeled()` 의 `svdb_data5.npz`(비트 주석만으로 RR)를 쓴다.

## 사전등록

| 관문 | 내용 | 문턱 |
|---|---|---|
| **Q7B-1** | TEST 채점 레코드 수 | **≥ 20** |
| **Q7B-2** | TEST **최대** 부트 SE (R12 — 근사 금지) | **≤ 0.10** |
| **Q7B-3** | TEST 지배 지분 (R11-3) | **≤ 50%** |
| **Q7B-4** | TEST 매크로 AUROC 의 **레코드 부트 95% CI 폭** | **≤ 0.10** |
| **Q7B-5** | 최대기여 개체를 빼도 매크로가 그 CI 안에 남는가 (R11-b) | 안에 있을 것 |

`GMIN` 은 **DEV 에서** 「최대 SE ≤ 0.10 **그리고** 중앙 SE ≤ 0.05」를 만족하는 **가장 작은**
값으로 정한다. DEV 에서 그런 GMIN 이 없으면 **거기서 종료**한다 — TEST 를 열지 않는다.

⚠️ 대조군으로 **V** 도 같이 낸다. Q1 에서 V 는 INCART GMIN=10 · 55명으로 통과했다.
V 가 통과하고 S 만 떨어지면 실패는 방법이 아니라 **S 의 희소성** 탓이다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/SCORING_RULES.md)
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    """사전등록 관문의 유일한 계약: 지지 / 기각 / **미결**."""
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def t_ci(v, conf=0.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2: return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

class AssetError(RuntimeError): pass
class LeakError(RuntimeError): pass
print("CELL 0 ✅ decide · t_ci 준비")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, time, subprocess, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0    = 20260803
SEEDS    = list(range(2000, 2005))     # 5시드 앙상블 (실험22-A 와 동일)
SE_CAP   = 0.05      # 레코드별 AUROC 부트 SE 의 **중앙값** 상한
SE_MAX   = 0.10      # **최대값** 상한 — R12: 근사 아니라 부트로 직접 잰다
N_MIN    = 20        # TEST 채점 레코드 하한
DOM_MAX  = 0.50      # TEST 안에서의 지배 지분 상한 (R11-3)
CIW_MAX  = 0.10      # 매크로 AUROC 레코드 부트 CI 폭 상한 (Q2 목표)
GMINS    = [2, 5, 10, 15, 20, 30, 40, 50, 75, 100, 150]
NB_BOOT  = 400       # 레코드별 SE 용
NB_MACRO = 4000      # 레코드 부트스트랩(매크로 CI)용
IDX_S, IDX_V = 1, 2

CONFIG = dict(
    exp="quest46_q7b_svdb_se", quest="ailab-2026-0046", step="svdb-se-check",
    parent_exp=["quest46_q7_svdb_cohort", "ailab-2026-0049",
                "quest46_q1_gmin_resolution", "ailab-2026-0047"],
    purpose=("Q7 은 양성 수만 셌다(GMIN=10 에서 66개). SE 는 예측이 있어야 잰다 — "
             "여기서 처음으로 학습이 들어간다. MIT-BIH DS1 학습 → SVDB zero-shot."),
    dataset="학습 MIT-BIH DS1 · 평가 SVDB(78레코드 · 레코드=환자 1:1) · 참고 MIT-BIH DS2",
    change_one_thing="평가 코호트만 INCART → SVDB. 백본·특징·에폭·손실·앙상블 동일",
    seeds=SEEDS,
    thresholds=dict(se_cap=SE_CAP, se_max=SE_MAX, n_min=N_MIN,
                    dom_max=DOM_MAX, ciw_max=CIW_MAX),
    contamination_policy={
        "train": "MIT-BIH DS1 만. SVDB 는 학습·튜닝·조기종료에 미사용. svdb_leak_audit 가 학습 전 강제",
        "gmin_selection": "SVDB 를 DEV/TEST 로 사전 분할. GMIN 은 **DEV 에서만** 고르고 관문은 **TEST 에서만**",
        "scaler": "RobustScaler·WST SelectKBest 모두 **DS1 에만** fit (wst_fit='ds1')",
        "threshold": "주 지표를 AUROC(순위 기반)로 둬 동작점 선택 오염을 원천 제거",
        "primary_arm": "raw(BN 미적응). BN 적응은 참고만 — 평가 입력을 쓰므로 transductive"},
    disclosed_limits=[
        "_medref·BN 적응은 평가 데이터의 **입력**을 쓴다(라벨 미사용 · transductive)",
        "코호트 선택 자체가 Q7 에서 SVDB **라벨 분포**를 보고 이뤄졌다",
        "SVDB 는 128Hz→360Hz 업샘플이라 대역폭이 64Hz 로 잘려 있다 — **전이 낙폭 인용 금지**. Q7-C 필요",
        "WST 를 DS1 에서만 fit 하므로 실험22-A(전체 fit)·PAPER §6.5 와 **직접 비교 불가**"],
    predictions={
        "Q7B-1": f"TEST 채점 레코드 ≥ {N_MIN}",
        "Q7B-2": f"TEST 최대 부트 SE ≤ {SE_MAX} (R12 — 근사 금지)",
        "Q7B-3": f"TEST 지배 지분 ≤ {DOM_MAX:.0%} (R11-3)",
        "Q7B-4": f"TEST 매크로 AUROC 레코드 부트 95% CI 폭 ≤ {CIW_MAX}",
        "Q7B-5": "최대기여 개체 제외값이 그 CI 안에 남는다 (R11-b)"},
    caveat=("DEV 에서 조건을 만족하는 GMIN 이 없으면 **TEST 를 열지 않고 종료**한다. "
            "V 를 대조군으로 같이 낸다 — V 가 통과하고 S 만 떨어지면 실패는 방법이 "
            "아니라 S 의 희소성 탓이다(Q1 에서 같은 구조를 봤다)"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7b_svdb_se", CONFIG, project=PROJECT)
run.log(f"설정 ✅ 시드 {SEEDS} · SE 상한 중앙 {SE_CAP}/최대 {SE_MAX} · N_MIN {N_MIN}")
run.log("🔒 GMIN 은 DEV 에서만 고른다. TEST 는 CELL 6 전까지 열지 않는다.")


In [ ]:
# CELL 2 — 【G0】 자산 점검 + **DEV/TEST 사전 분할** (예측 없음 · 학습 전)
#
#  ★ 여기서 분할을 확정하고 CONFIG 에 박는다. 예측을 만들기 **전에** 정해야
#    "SE 를 보고 갈랐다" 는 의심이 원천적으로 안 생긴다.
need = {os.path.join(MITBIH, "mamba_data.npz"): "MIT-BIH 비트·라벨·pid(학습원)",
        os.path.join(MITBIH, "colab_crossdb.py"): "교차DB 본체",
        os.path.join(MITBIH, "colab_crossdb_svdb.py"): "Q7-B 부록(이 실험 전용)",
        os.path.join(MITBIH, "svdb_labels.py"): "SVDB 라벨·오염없는 RR 빌더"}
miss = [p for p in need if not os.path.exists(p)]
if miss:
    raise AssetError("자산 없음:\n  " + "\n  ".join(f"{p} — {need[p]}" for p in miss)
                     + "\n  → repo 의 mit-bih/ 에서 Drive 로 올릴 것")
# ★ **존재만으로는 부족하다.** Drive 는 갱신 도구가 없어 실수로 빈 파일·자리표시자가
#   올라갈 수 있다(실제로 그런 적 있다). 내용에 필요한 심볼이 있는지까지 본다 —
#   없으면 여기서 크게 실패하는 편이 CELL 4 에서 NameError 로 죽는 것보다 낫다.
SIG = {"colab_crossdb.py": "def run_crossdb",
       "colab_crossdb_svdb.py": "def run_crossdb_svdb",
       "svdb_labels.py": "def build_labeled"}
for fn, sym in SIG.items():
    fp = os.path.join(MITBIH, fn); n = os.path.getsize(fp)
    if sym not in open(fp, encoding="utf-8", errors="replace").read():
        raise AssetError(f"{fn} 에 `{sym}` 가 없다 (크기 {n}B) — 자리표시자이거나 다른 파일이다.\n"
                         f"  → repo 의 mit-bih/{fn} 을 Drive 에 다시 올릴 것.\n"
                         f"  ⚠️ Drive 에 같은 이름이 **둘** 있으면 어느 쪽이 읽힐지 모른다 — 하나만 남길 것")
    run.log(f"  {fn:<26}{n:>9,}B  `{sym}` 확인 ✅")
run.log("자산 확인 ✅ " + " · ".join(os.path.basename(p) for p in need))

# ── Q7 이 만든 주석 카운트(예측 무관)를 재사용 ──
CNT_J = os.path.join(PROJECT, "data", "svdb_ann_counts.json")
if not os.path.exists(CNT_J):
    raise AssetError(f"{CNT_J} 없음 — Q7 노트북(quest46_q7_svdb_cohort)을 먼저 돌릴 것")
CNT = {int(k): {int(kk): vv for kk, vv in v.items()} for k, v in json.load(open(CNT_J)).items()}
SCOUNT = {r: c.get(1, 0) for r, c in CNT.items()}
run.log(f"\nQ7 주석 카운트 로드 — {len(SCOUNT)}레코드 · S 합계 {sum(SCOUNT.values()):,}")

# ── 결정론적 DEV/TEST 분할: S 개수 내림차순 정렬 후 **번갈아** 배정 ──
#    왜 번갈아: 무작위로 가르면 한쪽에 S 부담이 쏠려 GMIN 선택이 왜곡된다.
#    S 개수는 **주석**에서 온 것이고 예측과 무관하므로, 이걸로 층화해도 누수가 아니다.
#    ★ 1등(S 가 가장 많은 레코드)은 **TEST** 로 간다(order[0::2]). 그쪽이 지배 지분
#      관문(Q7B-3)을 **더 어렵게** 만든다 — 애매하면 보수적인 쪽으로 둔다. 반대로
#      두면 TEST 가 쉬워지고, 그건 결과를 좋게 만들려고 분할을 고른 것이 된다.
order = sorted(SCOUNT, key=lambda r: (-SCOUNT[r], r))     # 동점은 레코드 번호로 결정론
DEV_RECS  = sorted(order[1::2])
TEST_RECS = sorted(order[0::2])
assert not (set(DEV_RECS) & set(TEST_RECS)), "DEV/TEST 가 겹친다"
assert len(DEV_RECS) + len(TEST_RECS) == len(SCOUNT)
sdev  = sum(SCOUNT[r] for r in DEV_RECS)
stest = sum(SCOUNT[r] for r in TEST_RECS)
run.log(f"\n  🔒 사전 분할 확정 (예측 생성 전)")
run.log(f"    DEV  {len(DEV_RECS)}레코드 · S {sdev:,}  ← GMIN 은 여기서만 고른다")
run.log(f"    TEST {len(TEST_RECS)}레코드 · S {stest:,}  ← 관문은 여기서만 매긴다")
run.log(f"    S≥10 인 레코드: DEV {sum(SCOUNT[r]>=10 for r in DEV_RECS)}개"
        f" · TEST {sum(SCOUNT[r]>=10 for r in TEST_RECS)}개")
CONFIG["split"] = {"dev": DEV_RECS, "test": TEST_RECS,
                   "rule": "Q7 주석 S 개수 내림차순 정렬 후 번갈아 배정(동점은 레코드 번호). 예측 무관",
                   "s_dev": int(sdev), "s_test": int(stest)}
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【Q7B-P】 SVDB 준비 — **오염 없는 RR** 로 빌드 (오래 걸린다 · 캐시됨)
#
#  ★ svdb_prep.build_svdb() 의 `svdb_data.npz` 를 **쓰지 않는다.**
#    거기 pre_rr/post_rr 은 `rr = np.diff(ann.sample)` 로 만든 값이라 `'+'`(리듬변경)
#    주석이 낀 자리에서 **가짜 RR** 이 생긴다(svdb_labels.py 서두). S 는 RR 로
#    정의되는 클래스다 — 오염된 RR 로 S 를 재면 실험 자체가 무의미하다.
#
#  ⚠️ **함정**: Drive 에 `svdb_data.npz`(447MB, 07-27 빌드)가 **이미 있다.** 바로 그
#    오염된 파일이다. 있다고 갖다 쓰지 말 것 — 여기서는 이름부터 다른 `svdb_data5.npz`
#    만 본다. 특징 캐시도 `svdb_feats_q7b.npz` 로 분리했다(Drive 의 `svdb_feats/`
#    폴더는 그 오염 빌드 기준이라 정렬이 다를 수 있다).
SV5 = os.path.join(MITBIH, "svdb_data5.npz")
if os.path.exists(SV5):
    run.log(f"SVDB 빌드 캐시 적중 — {SV5}")
else:
    run.log("SVDB 신호 다운로드 + 빌드 (78레코드 · 수십 분). 재실행하면 캐시로 건너뛴다")
    g = globals()
    exec(open(os.path.join(MITBIH, "svdb_labels.py")).read(), g)
    g["build_labeled"](db="svdb", dldir=os.path.join(DRIVE_ROOT, "svdb_raw"), out=SV5)

_d5 = np.load(SV5, allow_pickle=True)
_k = _d5["y3"] >= 0
run.log(f"\n  비트 {int(_k.sum()):,} (F/Q 제외) · 레코드 {len(np.unique(_d5['pid'][_k]))}개")
run.log(f"  RR 은 **비트 주석만으로** 계산 — '+' 오염 없음")
run.log(f"  레코드 양끝 비트 {int(_d5['rr_edge'][_k].sum())}개는 RR 이 이웃 복제값이다(플래그 보존)")


In [ ]:
# CELL 4 — 【Q7B-T】 DS1 학습 → SVDB zero-shot 예측 (오염 감사가 학습 전에 돈다)
PROB_NPZ = run.data("q7b_svdb_probs_s5.npz")
if os.path.exists(PROB_NPZ):
    run.log(f"예측 캐시 적중 — {PROB_NPZ} (학습 건너뜀)")
    P = dict(np.load(PROB_NPZ))
else:
    g = globals()
    exec(open(os.path.join(MITBIH, "colab_crossdb.py")).read(), g)
    exec(open(os.path.join(MITBIH, "colab_crossdb_svdb.py")).read(), g)
    OUT = g["run_crossdb_svdb"](seeds=SEEDS, Kwst=40, wst_fit="ds1", use_rhythm=True)
    # R10: 확률과 **라벨·개체 id 를 같이** 저장한다. 길이만 맞는 걸로는 안 된다(R10-b).
    np.savez(PROB_NPZ, **{k: v for k, v in OUT.items() if isinstance(v, np.ndarray)})
    P = {k: v for k, v in OUT.items() if isinstance(v, np.ndarray)}
    run.log(f"\n예측 저장 → {PROB_NPZ}")

Y, REC = np.asarray(P["y_cross"]), np.asarray(P["rec_cross"])
if len(Y) != len(REC):
    raise LeakError(f"라벨 {len(Y)} vs 레코드id {len(REC)} 길이 불일치")
run.log(f"\n  SVDB 예측 {len(Y):,}비트 · 레코드 {len(np.unique(REC))}개"
        f" · N/S/V {np.bincount(Y, minlength=3)[:3].tolist()}")

# ── 분할이 예측 레코드를 실제로 덮는지 확인(사후에 바뀌지 않았음을 보임) ──
have = set(int(r) for r in np.unique(REC))
dev_have  = sorted(set(DEV_RECS)  & have)
test_have = sorted(set(TEST_RECS) & have)
run.log(f"  분할 대조 — DEV {len(dev_have)}/{len(DEV_RECS)} · TEST {len(test_have)}/{len(TEST_RECS)} 레코드에 예측 존재")
if not dev_have or not test_have:
    raise LeakError("DEV 또는 TEST 에 예측이 없다 — 분할과 빌드가 어긋났다")
if set(dev_have) & set(test_have):
    raise LeakError("DEV/TEST 가 겹친다")
CONFIG["n_pred"] = dict(beats=int(len(Y)), records=len(have),
                        dev=len(dev_have), test=len(test_have))
run.save_json("config", CONFIG)


In [ ]:
# CELL 4b — 【Q7B-L】 **빌드 손실 감사** (주석·레코드 존재만 본다 · 예측 미사용)
#
#  왜: Q7-A 는 주석에서 78레코드·S 12,198 을 셌다. CELL 3 의 빌드는 신호까지 받으므로
#  다운로드 실패·리드 부족으로 **레코드가 통째로 빠질 수 있다**. 빠진 게 하필 S-heavy
#  레코드면, 우리가 관문을 통과시킨 코호트는 Q7-A 가 심사한 코호트가 아니다.
#
#  ★ 이 셀은 **예측을 안 본다.** Q7-A 주석 카운트(분할 이전 정보)와 '어떤 레코드가
#    빌드에 들어왔나' 만 대조한다 — 선택에 쓸 수 있는 정보가 아니다(R10-c·R10-d).
built = set(int(r) for r in np.unique(REC))
allrec = set(SCOUNT)
lost = sorted(allrec - built, key=lambda r: -SCOUNT[r])
extra = sorted(built - allrec)          # ★ 반대 방향 — 이게 없으면 매핑 어긋남을 손실로 오독한다
run.log("\n" + "=" * 100)
run.log("【Q7B-L】 빌드 대조 감사 — Q7-A 주석 대비 (양방향)")
run.log("=" * 100)
run.log(f"  Q7-A 주석 {len(allrec)}레코드 · S {sum(SCOUNT.values()):,}")
run.log(f"  빌드 레코드 {len(built)}개")
run.log(f"  교집합 {len(allrec & built)} · 주석에만 {len(lost)} · **빌드에만 {len(extra)}**")
if extra:
    # ★ 빌드에만 있는 ID 가 있으면 이건 '손실' 이 아니라 **매핑 어긋남**이다.
    #   손실이면 빌드 ⊆ 주석 이어야 하고 빌드 개수가 주석보다 적어야 한다.
    run.log(f"\n  ⛔ **손실이 아니라 ID 매핑 어긋남이다.** 빌드에만 있는 ID: {extra[:12]}"
            + (" ..." if len(extra) > 12 else ""))
    run.log(f"     손실이라면 빌드 ⊆ 주석 이어야 하는데 아니다"
            f" (빌드 {len(built)} vs 주석 {len(allrec)}).")
    run.log("     → 아래 【Q7B-M】 매핑 검증을 반드시 돌릴 것. 그 전에는 레코드 번호를 인용하지 않는다.")
    CONFIG["build_loss"] = {"kind": "id_mismatch", "n_ann": len(allrec), "n_built": len(built),
                            "only_ann": [int(r) for r in lost], "only_built": [int(r) for r in extra]}
    run.save_json("config", CONFIG)
elif not lost:
    run.log("  ✅ 손실 없음")
else:
    slost = sum(SCOUNT[r] for r in lost)
    run.log(f"  ⚠️ **{len(lost)}레코드 손실** · 그 안의 S {slost:,}"
            f" ({slost/max(sum(SCOUNT.values()),1):.1%})")
    run.log(f"    손실 레코드(S 많은 순): "
            + ", ".join(f"#{r}({SCOUNT[r]:,})" for r in lost[:12])
            + (" ..." if len(lost) > 12 else ""))
    # R10-c: 손실이 S-heavy 쪽에 쏠렸나? 전체 대비 손실군의 S 분포를 비교한다.
    alls = np.array(sorted(SCOUNT.values(), reverse=True))
    ls = np.array(sorted((SCOUNT[r] for r in lost), reverse=True))
    run.log(f"    S 중앙값 — 전체 {np.median(alls):g} vs 손실군 {np.median(ls):g}")
    run.log(f"    손실군 최대 S {ls.max():,}"
            + ("  ← **최상위 레코드가 빠졌다**" if ls.max() == alls.max() else ""))
    if ls.max() == alls.max():
        run.log("    ⛔ 지배 개체가 빠졌으므로 **지배 지분·커버리지는 낙관적으로 보인다.**")
        rest = sum(SCOUNT[r] for r in built)
        run.log(f"       복구 시 전체 지배 지분 ≈ {ls.max()/max(rest+ls.max(),1):.1%}"
                f" (현재 빌드 기준 {max(SCOUNT[r] for r in built)/max(rest,1):.1%})")
    # 분할별로도 본다 — 한쪽에 쏠렸으면 DEV/TEST 비교가 왜곡된다
    ld = [r for r in lost if r in DEV_RECS]; lt = [r for r in lost if r in TEST_RECS]
    run.log(f"    분할별 손실 — DEV {len(ld)}/{len(DEV_RECS)} (S {sum(SCOUNT[r] for r in ld):,})"
            f" · TEST {len(lt)}/{len(TEST_RECS)} (S {sum(SCOUNT[r] for r in lt):,})")
    run.log("  → CELL 3 로그에서 `✗` 줄을 확인할 것(다운로드 실패인지 리드 부족인지).")
    run.log("     복구 가능하면 `svdb_data5.npz` 를 지우고 CELL 3 을 다시 돌린다.")
CONFIG["build_loss"] = {"n_ann": len(allrec), "n_built": len(built),
                        "lost": [int(r) for r in lost],
                        "lost_s": int(sum(SCOUNT[r] for r in lost)),
                        "top_record_lost": bool(lost and SCOUNT[lost[0]] == max(SCOUNT.values()))}
run.save_json("config", CONFIG)


In [ ]:
# CELL 4c — 【Q7B-M】 **ID 매핑 검증·복원** — 레코드 번호를 인용하기 전에 반드시 통과할 것
#
#  실제로 사고가 났다(ailab-2026-0052). 원인은 `colab_crossdb_svdb._svdb_load` 의
#  **조용한 fallback** 이었다:
#      try:    recs = wfdb.get_record_list("svdb")        # 실제 78개(빈 번호 있음)
#      except: recs = [str(i) for i in range(800, 895)]   # 연속 95개  ← 이게 발동
#  그러면 `recs[i] = 800+i` 라 **라벨 = 800 + 인덱스** 가 된다. SVDB 는 813~819·830~839
#  가 비어 있어 인덱스 13 부터 어긋나고, 존재하지 않는 번호 17개가 라벨로 찍힌다.
#
#  ★ 복원은 계수 짝짓기가 아니라 **결정론적**이다: 참 번호 = recs[라벨 − 800].
#    그런 다음 (N,S,V) 삼중값으로 **검증**한다(짝짓기가 아니라 확인).
import wfdb
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # fallback 없음 — 실패하면 터진다
run.log("\n" + "=" * 100)
run.log("【Q7B-M】 ID 매핑 검증·복원")
run.log("=" * 100)
run.log(f"  실제 SVDB 레코드 {len(recs)}개 · 범위 {min(recs)}~{max(recs)}")
holes = [n for n in range(min(recs), max(recs) + 1) if n not in recs]
run.log(f"  빈 번호 {len(holes)}개: {holes}")

labels = sorted(int(x) for x in np.unique(REC))
outside = [l for l in labels if l not in recs]
CONT = [min(recs) + i for i in range(len(recs))]           # 연속 가정 라벨
SIG = (labels == CONT)                                     # fallback 사고의 지문

def counts(lab):
    m = REC == lab
    return tuple(int((Y[m] == k).sum()) for k in range(3))

if not outside:
    run.log("\n  ✅ 라벨이 전부 실제 목록 안에 있다 — 매핑 정상")
    MAP = {l: l for l in labels}
elif SIG:
    run.log(f"\n  ⛔ **fallback 사고 지문 확인.** 라벨이 정확히 연속 {CONT[0]}~{CONT[-1]} 이고")
    run.log(f"     그중 {len(outside)}개가 존재하지 않는 번호다: {outside}")
    MAP = {l: recs[l - CONT[0]] for l in labels}
    run.log(f"     → 결정론적 복원: 참 번호 = recs[라벨 − {CONT[0]}]")
else:
    raise AssetError(f"라벨 {len(outside)}개가 목록 밖인데 연속 가정과도 안 맞는다: "
                     f"{outside[:8]} — 원인 미상. 수동 확인 필요")

# ── 검증: 복원한 번호의 (N,S,V) 가 Q7-A 주석과 맞는가 (짝짓기가 아니라 **확인**)
CNT3 = {int(k): (v.get(0, 0), v.get(1, 0), v.get(2, 0)) for k, v in CNT.items()}
agree, bad = 0, []
for l in labels:
    t = MAP[l]; o = counts(l); a = CNT3.get(t)
    if a and all(abs(x - y) <= 2 for x, y in zip(o, a)):
        agree += 1
    else:
        bad.append((l, t, o, a))
run.log(f"\n  검증 — 복원 번호의 (N,S,V) 가 주석과 ±2 안에서 일치: **{agree}/{len(labels)}**")
if bad[:6]:
    run.log(f"  {'라벨':>6}{'참':>6}   예측(N,S,V)            주석(N,S,V)")
    for l, t, o, a in bad[:6]:
        run.log(f"  {l:>6}{t:>6}   {str(o):<22}{a}")
MAP_OK = agree >= 0.9 * len(labels)
run.log(f"  → 매핑 {'✅ 확정' if MAP_OK else '❌ 미확정 — 레코드 번호 인용 금지'}")

D = CONFIG.get("result", {}).get("test_S")
if MAP_OK and outside and D:
    run.log("\n  ── 지금까지 인용한 번호의 **참 정체** ──")
    order = list(np.argsort(np.array(D["auroc"])))[:4] + [int(np.argmax(D["pos"]))]
    for i in dict.fromkeys(order):
        l = int(D["recs"][i])
        run.log(f"    #{l} → **#{MAP.get(l, l)}**   AUROC {D['auroc'][i]:.4f} · S {D['pos'][i]:,}")
if MAP_OK and outside:
    run.log("\n  ⚠️ **무사한 것**: 비트 묶음은 같은 인덱스끼리 되므로 레코드별 AUROC·SE·")
    run.log("     매크로·CI·지배지분은 그대로 유효하다. DEV/TEST 도 서로소이고 예측 전에")
    run.log("     정해졌으므로 **오염 차단·관문 판정은 깨지지 않았다.**")
    run.log("  ⚠️ **무너진 것**: ① S 개수로 층화했다는 주장 — 라벨이 어긋난 채 층화했으니")
    run.log("     실제로는 임의 분할이다(DEV/TEST 불균형의 유력한 원인). ② 「1등 레코드를")
    run.log("     TEST 로 보내 보수적으로 만들었다」 는 주장도 성립하지 않는다.")

json.dump({str(k): int(v) for k, v in MAP.items()},
          open(os.path.join(PROJECT, "data", "svdb_id_map_q7b.json"), "w"))
CONFIG["id_map"] = {"ok": bool(MAP_OK),
                    "signature": "clean" if not outside else ("contiguous_fallback" if SIG else "unknown"),
                    "n_outside": len(outside), "agree": agree, "n": len(labels),
                    "map": {str(k): int(v) for k, v in MAP.items()}}
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【Q7B-A】 **DEV 에서만** GMIN 을 고른다 (TEST 는 아직 안 연다)
#
#  R12: "검증이 선택에 의존하게 짜지 않는다." GMIN 을 훑어서 SE 가 통과하는 값을
#  고르면 그건 테스트로 고른 것이다. 그래서 고르는 곳(DEV)과 매기는 곳(TEST)을
#  물리적으로 가른다.
from sklearn.metrics import roc_auc_score

def per_record(score, y, rec, recs, idx, nboot=NB_BOOT, seed=SEED0):
    """레코드별 (AUROC, 부트 SE, 양성 수). SE 는 **부트로 직접** 잰다(R12)."""
    rng = np.random.RandomState(seed); out = {}
    for r in recs:
        m = np.where(rec == r)[0]
        if len(m) < 3: continue
        t = (y[m] == idx); s = score[m]
        if not (t.any() and not t.all()): continue
        vals = []
        for _ in range(nboot):
            j = rng.randint(0, len(m), len(m)); tj = t[j]
            if 0 < tj.sum() < len(tj):
                vals.append(roc_auc_score(tj.astype(int), s[j]))
        out[int(r)] = (float(roc_auc_score(t.astype(int), s)),
                       float(np.std(vals, ddof=1)) if len(vals) > 2 else np.nan,
                       int(t.sum()))
    return out

SC_S = P["v2_cross_raw"].mean(0)[:, IDX_S]      # raw = BN 미적응 (주 arm)
SC_V = P["v2_cross_raw"].mean(0)[:, IDX_V]

run.log("\n" + "=" * 100)
run.log("【Q7B-A】 DEV 에서 GMIN 선택 — TEST 는 봉인 상태")
run.log("=" * 100)
PR_DEV = per_record(SC_S, Y, REC, dev_have, IDX_S)
run.log(f"  DEV 채점 가능 {len(PR_DEV)}/{len(dev_have)}레코드 (양성·음성이 모두 있는 곳)")

run.log(f"\n  {'GMIN':>5}{'레코드':>7}{'중앙SE':>9}{'최대SE':>9}{'커버':>8}   판정")
CAND = None; ROWS = []
for gm in GMINS:
    keep = [r for r, (a, se, p) in PR_DEV.items() if p >= gm and np.isfinite(se)]
    if not keep:
        ROWS.append(dict(gmin=gm, n=0)); continue
    se = np.array([PR_DEV[r][1] for r in keep]); pos = np.array([PR_DEV[r][2] for r in keep])
    tot = sum(p for _, _, p in PR_DEV.values())
    med, mx, cov = float(np.median(se)), float(se.max()), float(pos.sum() / max(tot, 1))
    ok = (mx <= SE_MAX) and (med <= SE_CAP)
    ROWS.append(dict(gmin=gm, n=len(keep), se_med=med, se_max=mx, cov=cov, ok=bool(ok)))
    run.log(f"  {gm:>5}{len(keep):>7}{med:>9.4f}{mx:>9.4f}{cov:>8.1%}   {'✅' if ok else '—'}")
    if ok and CAND is None:
        CAND = gm          # **가장 작은** 통과 GMIN (개체를 최대한 남긴다)

CONFIG["dev_sweep"] = ROWS
if CAND is None:
    run.log(f"\n  ❌ DEV 에서 「최대SE ≤ {SE_MAX} 그리고 중앙SE ≤ {SE_CAP}」를 만족하는 GMIN 이 없다.")
    run.log("     사전등록대로 **TEST 를 열지 않고 종료**한다. GMIN 을 더 올려 찾는 것은")
    run.log("     '통과하는 값 고르기' 이므로 금지 — 필요하면 새 사전등록으로 다시 돌린다.")
    CONFIG["result"] = {"verdicts": {k: "❌ 기각" for k in
                                     ("Q7B-1", "Q7B-2", "Q7B-3", "Q7B-4", "Q7B-5")},
                        "stopped_at": "dev", "gmin": None}
    run.save_json("config", CONFIG)
else:
    run.log(f"\n  🔒 **GMIN = {CAND}** 로 확정 (DEV 에서만 보고 정했다). 이제 TEST 를 연다.")
    CONFIG["gmin_selected"] = CAND
    run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【Q7B-B】 **TEST 채점** — 사전등록 관문 (R11-b 보고 항목 포함)
if CONFIG.get("gmin_selected") is None:
    run.log("DEV 에서 종료됐다 — TEST 를 열지 않는다.")
else:
    GM = CONFIG["gmin_selected"]
    run.log("\n" + "=" * 100)
    run.log(f"【Q7B-B】 TEST 채점 · GMIN={GM} (DEV 에서 고른 값)")
    run.log("=" * 100)

    def score_cohort(score, idx, recs, tag):
        PR = per_record(score, Y, REC, recs, idx)
        keep = sorted([r for r, (a, se, p) in PR.items() if p >= GM and np.isfinite(se)])
        if len(keep) < 2:
            run.log(f"  ⚠️ {tag}: 채점 개체 {len(keep)}개 — 매크로 성립 불가")
            return None
        a = np.array([PR[r][0] for r in keep]); se = np.array([PR[r][1] for r in keep])
        pos = np.array([PR[r][2] for r in keep])
        rng = np.random.RandomState(SEED0 + 7)
        bs = [float(a[rng.randint(0, len(a), len(a))].mean()) for _ in range(NB_MACRO)]
        lo, hi = np.percentile(bs, [2.5, 97.5])
        # R11-b: 최대기여 개체 = 빼면 매크로가 가장 많이 움직이는 개체
        drops = np.array([float(np.delete(a, i).mean()) for i in range(len(a))])
        j = int(np.argmax(np.abs(drops - a.mean())))
        return dict(tag=tag, recs=keep, auroc=a.tolist(), se=se.tolist(), pos=pos.tolist(),
                    n=len(keep), macro=float(a.mean()), lo=float(lo), hi=float(hi),
                    width=float(hi - lo), se_med=float(np.median(se)), se_max=float(se.max()),
                    dom=float(pos.max() / max(pos.sum(), 1)), cov=float(pos.sum() / max(sum(p for _, _, p in PR.values()), 1)),
                    drop_rec=int(keep[j]), drop_macro=float(drops[j]))

    # 대조군 V 도 **같은 GMIN** 을 쓴다. V 는 양성이 훨씬 많아 GM 이 거의 안 깎으므로
    # 사실상 전수다 — 'S 만 떨어졌나' 를 보려는 것이지 V 를 최적화하려는 게 아니다.
    S_TEST = score_cohort(SC_S, IDX_S, test_have, "TEST · S")
    V_TEST = score_cohort(SC_V, IDX_V, test_have, "TEST · V (대조군)")
    S_DEV_AT_GM = score_cohort(SC_S, IDX_S, dev_have, "DEV · S (참고)")

    run.log(f"\n  {'코호트':<22}{'개체':>5}{'매크로':>9}{'레코드 부트 95% CI':>24}{'폭':>8}"
            f"{'중앙SE':>9}{'최대SE':>9}{'지배':>8}")
    for D in (S_TEST, V_TEST, S_DEV_AT_GM):
        if D:
            run.log(f"  {D['tag']:<22}{D['n']:>5}{D['macro']:>9.4f}"
                    f"   [{D['lo']:.4f}, {D['hi']:.4f}]{D['width']:>8.4f}"
                    f"{D['se_med']:>9.4f}{D['se_max']:>9.4f}{D['dom']:>8.1%}")

    VERD = {}
    def g_(k, ok, d):
        VERD[k] = "✅ 지지" if ok else "❌ 기각"; run.log(f"  {k:<8}{VERD[k]}  {d}")

    run.log("")
    if S_TEST is None:
        for k in ("Q7B-1", "Q7B-2", "Q7B-3", "Q7B-4", "Q7B-5"): VERD[k] = "❌ 기각"
        run.log("  ❌ TEST 에 매크로가 안 선다 — 전 관문 기각")
    else:
        D = S_TEST
        g_("Q7B-1", D["n"] >= N_MIN, f"TEST 채점 레코드 **{D['n']}** ≥ {N_MIN}")
        g_("Q7B-2", D["se_max"] <= SE_MAX,
           f"TEST **최대** 부트 SE **{D['se_max']:.4f}** ≤ {SE_MAX}"
           f"  (중앙 {D['se_med']:.4f} · R12 대로 부트로 직접 쟀다)")
        g_("Q7B-3", D["dom"] <= DOM_MAX, f"TEST 지배 지분 **{D['dom']:.1%}** ≤ {DOM_MAX:.0%}")
        g_("Q7B-4", D["width"] <= CIW_MAX,
           f"매크로 AUROC **{D['macro']:.4f}** [{D['lo']:.4f}, {D['hi']:.4f}] 폭 **{D['width']:.4f}** ≤ {CIW_MAX}")
        g_("Q7B-5", D["lo"] <= D["drop_macro"] <= D["hi"],
           f"최대기여 #{D['drop_rec']} 제외 매크로 **{D['drop_macro']:.4f}** 가 CI 안에 있나 (R11-b)")
        run.log(f"\n  개체별 값 전량 (R11-b) — GMIN={GM}")
        for r, a, se, p in zip(D["recs"], D["auroc"], D["se"], D["pos"]):
            run.log(f"    #{r}  AUROC {a:.4f}  SE {se:.4f}  S {p:>5,}"
                    + ("   ← 최대기여" if r == D["drop_rec"] else ""))

    # ── R15: 매크로 CI 폭은 **무엇이** 만드나 — 측정오차인가 개체 간 이질성인가
    if S_TEST:
        av = np.array(S_TEST["auroc"]); sv = np.array(S_TEST["se"]); nn = len(av)
        w_het = 2 * 1.96 * av.std(ddof=1) / np.sqrt(nn)
        w_err = 2 * 1.96 * np.sqrt((sv ** 2).mean()) / np.sqrt(nn)
        run.log(f"\n  ── CI 폭 분해 (R15) ──")
        run.log(f"    관측 폭 {S_TEST['width']:.4f}")
        run.log(f"    개체 간 이질성만으로 {w_het:.4f}  (개체 간 AUROC SD {av.std(ddof=1):.4f})")
        run.log(f"    측정오차만으로     {w_err:.4f}  (RMS SE {np.sqrt((sv**2).mean()):.4f})")
        run.log(f"    분산비 이질성/측정오차 = {av.var(ddof=1) / max((sv**2).mean(), 1e-12):.1f}배")
        run.log("    → 매크로 CI 를 좁히는 건 **개체를 늘리는 것**이지 GMIN 을 올리는 게 아니다.")
        CONFIG.setdefault("ciw_decomp", {}).update(
            dict(observed=S_TEST["width"], het_only=float(w_het), err_only=float(w_err),
                 var_ratio=float(av.var(ddof=1) / max((sv ** 2).mean(), 1e-12))))

    run.log("\n  " + "  ".join(f"{k}: {v}" for k, v in VERD.items()))
    if V_TEST:
        run.log(f"\n  대조군 V — 개체 {V_TEST['n']} · 매크로 {V_TEST['macro']:.4f}"
                f" · 최대SE {V_TEST['se_max']:.4f}")
        if S_TEST and V_TEST["se_max"] <= SE_MAX and S_TEST["se_max"] > SE_MAX:
            run.log("    → V 는 통과하고 S 만 떨어졌다. 실패는 **방법이 아니라 S 의 희소성** 탓이다"
                    "(Q1 에서 본 구조와 같다).")
    run.log("\n  ⚠️ 전이 낙폭은 여기서 인용하지 않는다 — SVDB 는 128→360Hz 업샘플이라")
    run.log("     대역폭이 64Hz 로 잘려 있다. 낙폭이 필요하면 Q7-C(동일 대역 대조군).")
    run.log("  ⚠️ 이 값들은 WST 를 DS1 에서만 fit 했으므로 실험22-A·§6.5 와 직접 비교 불가.")
    CONFIG["result"] = {"verdicts": VERD, "gmin": GM, "stopped_at": "test",
                        "test_S": S_TEST, "test_V": V_TEST, "dev_S_at_gmin": S_DEV_AT_GM}
    run.save_json("config", CONFIG)


In [ ]:
# CELL 6b — 【Q7B-X】 **탐색적 부지표** — 사후등록이 아니다
#
#  ★ 관문 판정(Q7B-1~5)은 CELL 6 에서 **AUROC 로 이미 확정**됐고 여기서 바꾸지 않는다.
#    아래는 그 판정에 딸린 **해석 재료**다. 학습 0회 — 저장된 확률만 다시 읽는다.
#
#  왜 필요한가 — 07-17 전례:
#      S ROC-AUC 0.9223 → 0.9020  (Δ −0.020 · "애매")
#      S PR-AUC  0.5138 → 0.2944  (Δ −0.219 · "결정적")
#    ROC 로는 거의 아무 일도 없어 보이는데 PR 로는 반토막이었다. 그때 남긴 교훈이
#    "극단 불균형에선 PR-AUC 를 primary 로" 였다. Q7-B 가 AUROC 를 primary 로 쓴 건
#    **동작점 오염을 원천 차단하려는** 선택이고 그 자체는 타당하다. 다만 그 대가를
#    기록하지 않으면 "AUROC 0.926" 이 정밀도까지 보증하는 것처럼 읽힌다. 안 읽힌다.
#
#  ⚠️ PR-AUC 는 **유병률에 붙은 지표**다(R4). 레코드마다 S 유병률이 다르므로 PR-AUC
#     원값을 레코드끼리·코호트끼리 비교하면 안 된다. 그래서 **유병률과 우연수준 대비
#     배수(lift)** 를 반드시 같이 낸다.
from sklearn.metrics import average_precision_score
from scipy import stats as _st

if CONFIG.get("result", {}).get("stopped_at") != "test" or not CONFIG["result"].get("test_S"):
    run.log("TEST 채점이 없다 — 부지표 생략")
else:
    _V0 = dict(CONFIG["result"]["verdicts"])           # 관문 불변 확인용 스냅샷
    D = CONFIG["result"]["test_S"]; GM = CONFIG["result"]["gmin"]
    run.log("\n" + "=" * 100)
    run.log("【Q7B-X】 탐색적 부지표 — 관문은 AUROC 로 확정된 채 그대로 둔다")
    run.log("=" * 100)

    rows = []
    for r in D["recs"]:
        m = np.where(REC == r)[0]; t = (Y[m] == IDX_S).astype(int); s = SC_S[m]
        prev = float(t.mean())
        ap = float(average_precision_score(t, s))
        rows.append(dict(rec=int(r), auroc=float(D["auroc"][D["recs"].index(r)]),
                         pos=int(t.sum()), prev=prev, pr=ap, lift=ap / max(prev, 1e-9)))
    au = np.array([x["auroc"] for x in rows]); pr = np.array([x["pr"] for x in rows])
    pos = np.array([x["pos"] for x in rows]); prev = np.array([x["prev"] for x in rows])
    lift = np.array([x["lift"] for x in rows])

    run.log(f"\n  레코드 매크로 — AUROC {au.mean():.4f} · **PR-AUC {pr.mean():.4f}**"
            f" · 우연수준(평균 유병률) {prev.mean():.4f} · lift {lift.mean():.1f}배")
    rng = np.random.RandomState(SEED0 + 11)
    bs = [float(pr[rng.randint(0, len(pr), len(pr))].mean()) for _ in range(NB_MACRO)]
    plo, phi = np.percentile(bs, [2.5, 97.5])
    run.log(f"    PR-AUC 레코드 부트 95% CI [{plo:.4f}, {phi:.4f}] 폭 {phi-plo:.4f}")
    run.log(f"    ⚠️ 07-17 의 PR-AUC 0.514/0.294 와 **직접 비교 금지** — 그건 DS2 pooled 값이고"
            f" 유병률이 다르다(R4). 여기 유병률 중앙 {np.median(prev):.4f}")

    # ── ① S 부담과 성능의 관계 — 매크로가 무엇을 가리고 있나
    #  ★ 원 PR-AUC 를 군끼리 비교하면 **안 된다**(R4). 군마다 유병률이 다르고 PR-AUC 의
    #    우연수준이 곧 유병률이다. 그래서 유병률·lift·정규화PR 을 같이 낸다.
    #      정규화PR = (AP − 유병률) / (1 − 유병률)   ← 우연 0, 완벽 1 (유병률과 무관)
    npr = (pr - prev) / np.maximum(1 - prev, 1e-9)
    run.log(f"\n  ── S 부담별 성능 ── (원 PR-AUC 는 군끼리 비교 금지 — R4)")
    run.log(f"  {'구간':<14}{'개체':>5}{'AUROC':>9}{'PR-AUC':>9}{'유병률':>9}"
            f"{'lift':>8}{'천장':>8}{'정규화PR':>10}{'보유 S':>9}{'지분':>8}")
    for lo, hi, tag in ((100, 10**9, "S ≥ 100"), (50, 100, "50 ≤ S < 100"), (0, 50, "S < 50")):
        m = (pos >= lo) & (pos < hi)
        if m.sum():
            run.log(f"  {tag:<14}{int(m.sum()):>5}{au[m].mean():>9.4f}{pr[m].mean():>9.4f}"
                    f"{prev[m].mean():>9.4f}{lift[m].mean():>8.1f}"
                    f"{1/max(prev[m].mean(),1e-9):>8.1f}{npr[m].mean():>10.3f}"
                    f"{int(pos[m].sum()):>9,}{pos[m].sum()/pos.sum():>8.1%}")
    run.log(f"  {'전체':<14}{len(au):>5}{au.mean():>9.4f}{pr.mean():>9.4f}"
            f"{prev.mean():>9.4f}{lift.mean():>8.1f}{1/max(prev.mean(),1e-9):>8.1f}"
            f"{npr.mean():>10.3f}{int(pos.sum()):>9,}{1.0:>8.1%}")
    #  ★★ 실측이 내 처방을 반증했다. 처음에 "정규화PR·lift 로 읽으면 된다" 고 썼는데
    #     **셋 다 유병률에서 자유롭지 않다**:
    #       · lift = AP/유병률 은 천장이 **1/유병률** 이다. S≥100 군은 천장의 92.7% 로
    #         이미 포화 — '낮다' 가 아니라 '더 올라갈 데가 없다' 다.
    #       · 정규화PR = (AP−π)/(1−π) 도 π→0 이면 함께 0 으로 간다(고정 ROC 에서
    #         AP ≈ cπ 이므로 정규화PR ≈ π(c−1)). 실측에서도 원 PR 과 **같은 순서**였다.
    #       · 유병률이 다른 개체들의 lift 를 평균하면 무의미하다 — 아래 자기검증 참조.
    #     → **개체·군을 비교하는 지표는 AUROC 뿐이다**(순위만의 함수라 유병률 무관).
    #       AP·유병률은 '그 개체를 서술' 하는 데 쓰고, 군 순위를 매기는 데 쓰지 않는다.
    if (pos >= 100).any() and (pos < 50).any() \
            and pr[pos >= 100].mean() > pr[pos < 50].mean() \
            and au[pos >= 100].mean() < au[pos < 50].mean():
        run.log("    ⛔ **AUROC 와 PR 계열이 군 순서를 반대로 매긴다.**")
        run.log("       부담 큰 군은 **순위 품질이 나쁜데(AUROC↓) 정밀도는 높다(PR↑)**.")
        run.log("       모순이 아니다 — 양성이 흔하면 어설픈 순위로도 정밀도가 나온다.")
        run.log("       ⚠️ 정규화PR 도 이 순서를 못 뒤집는다(π→0 이면 같이 0 으로 간다).")
        run.log("       ⚠️ lift 는 천장이 1/유병률 이라 부담 큰 군이 구조적으로 불리하다.")
        run.log("       → **군·개체 비교는 AUROC 로만 한다.** AP·유병률은 서술용이다.")
    # lift 평균의 자기검증: 유병률이 다른 개체를 섞으면 물리적으로 불가능한 값이 나온다
    if lift.mean() * prev.mean() > 1.0:
        run.log(f"    ⛔ 자기검증 실패 — 평균 lift {lift.mean():.1f} × 평균 유병률 "
                f"{prev.mean():.4f} = {lift.mean()*prev.mean():.2f} > 1. 한 개체에서는 "
                f"불가능한 값이다. **유병률이 다른 개체의 lift 를 평균하지 말 것.**")
    rho, prho = _st.spearmanr(pos, au)
    run.log(f"    양성수 vs AUROC 스피어만 rho={rho:+.3f} p={prho:.4f}")
    wmean = float(np.average(au, weights=pos))
    run.log(f"    단순 매크로 {au.mean():.4f} vs 양성수 가중 {wmean:.4f} (차 {au.mean()-wmean:+.4f})")
    run.log("    ⛔ **가중값으로 갈아타지 않는다.** `w=n_pos` 는 최대가중이 정의상 지배")
    run.log("       지분과 같아(R14-b ②) 매크로를 전역 지표 쪽으로 되감는다 — R11 이 피하려던")
    run.log("       방향이다. 두 값이 갈린다는 **사실 자체**가 결과다(→ R11-c).")

    # ── ② R11-b 최대기여: 두 정의를 **둘 다** 낸다
    drops = np.array([float(np.delete(au, k).mean()) for k in range(len(au))])
    j_rm = int(np.argmax(np.abs(drops - au.mean())))
    infl = np.abs(au - au.mean()) * pos
    j_w = int(np.argmax(infl))
    run.log(f"\n  ── 최대기여 개체: 두 정의 (→ R11-c) ──")
    run.log(f"    제거영향 최대   #{rows[j_rm]['rec']}  AUROC {au[j_rm]:.4f} · S {pos[j_rm]:>4}"
            f" · 제외 매크로 {drops[j_rm]:.4f} (Δ {drops[j_rm]-au.mean():+.4f})")
    run.log(f"    부담가중 영향   #{rows[j_w]['rec']}  AUROC {au[j_w]:.4f} · S {pos[j_w]:>4}"
            f" · |편차|×S = {infl[j_w]:,.0f}")
    run.log(f"    ⚠️ 개체가 {len(au)}개면 하나를 빼도 매크로는 1/{len(au)} 만큼밖에 안 움직인다."
            f" 실제 Δ {abs(drops[j_rm]-au.mean()):.4f} — **Q7B-5 는 이 n 에서 거의 자동 통과**다.")
    run.log(f"       R11-b 를 낳은 #213 사건은 개체가 7개였다. 관문의 힘이 n 에 의존한다(R11-c).")

    # ── ③ 진짜 실패 레코드 — 측정 불안정이 아니다
    sev = np.array([D["se"][D["recs"].index(x["rec"])] for x in rows])
    bad = [k for k in np.argsort(au)[:6] if pos[k] >= 100]
    if bad:
        run.log(f"\n  ── 실패 레코드(양성 100+ 인데 AUROC 하위) — SE 가 작으니 **진짜 실패**다 ──")
        run.log(f"    {'':7}{'AUROC':>8}{'PR-AUC':>9}{'유병률':>9}{'lift':>7}{'천장':>7}{'천장대비':>9}{'S':>7}{'SE':>9}")
        for k in bad:
            run.log(f"    #{rows[k]['rec']:<6}{au[k]:>8.4f}{pr[k]:>9.4f}{prev[k]:>9.4f}"
                    f"{lift[k]:>7.2f}{1/max(prev[k],1e-9):>7.1f}{lift[k]*prev[k]:>9.1%}"
                    f"{pos[k]:>7,}{sev[k]:>9.4f}")
        run.log("    ⚠️ 이들의 lift 를 코호트 평균 lift 와 비교하지 않는다 — 천장이 다르다"
                " (여기 유병률이 높아 천장 자체가 5~11배뿐이다).")
        run.log("       **심각도 판정은 AUROC 로 한다.** AP·유병률은 '이 개체가 어떤 상황인가' 를 말할 뿐이다.")
        run.log("    → 다음 진단 표적. MIT-BIH #200(P파 소실형)·#222(기준선 오염형)와")
        run.log("      같은 유형인지 새 유형인지 가른다. 재학습 불필요.")

    # ── ④ within(DS2) 맥락 — 같은 모델·같은 채점. **빼지 않는다**
    if "v2_within_raw" in P and "rec_within" in P:
        yw = np.asarray(P["y_within"]); rw = np.asarray(P["rec_within"])
        sw = P["v2_within_raw"].mean(0)[:, IDX_S]
        wa = []
        for r in np.unique(rw):
            m = np.where(rw == r)[0]; t = (yw[m] == IDX_S)
            if t.sum() >= GM and t.any() and not t.all():
                wa.append(roc_auc_score(t.astype(int), sw[m]))
        if len(wa) >= 2:
            run.log(f"\n  ── 맥락: within DS2 (같은 모델·같은 채점) ──")
            run.log(f"    개체 {len(wa)} · 매크로 AUROC {np.mean(wa):.4f}")
            run.log("    ⛔ **SVDB 값에서 빼지 말 것.** SVDB 는 128→360Hz 업샘플이라 대역이")
            run.log("       다르다 — 낙폭에 모델 요인과 대역 요인이 섞인다. 그건 Q7-C 다.")
            run.log(f"    ⚠️ DS2 는 S 채점 개체가 구조적으로 적다(Q1: 상한 13명) — {len(wa)}개.")

    CONFIG["exploratory"] = {
        "note": "관문 아님. AUROC 판정은 CELL 6 에서 확정된 채 유지된다",
        "pr_auc_macro": float(pr.mean()), "pr_auc_ci": [float(plo), float(phi)],
        "prev_median": float(np.median(prev)), "lift_macro": float(lift.mean()),
        "auroc_weighted_by_pos": wmean, "spearman_pos_auroc": [float(rho), float(prho)],
        "npr_macro": float(npr.mean()),
        "max_removal_rec": rows[j_rm]["rec"], "max_burden_rec": rows[j_w]["rec"],
        "per_record": rows}
    assert CONFIG["result"]["verdicts"] == _V0, "부지표 셀이 관문을 바꿨다 — 사후등록이다"
    run.log("\n  ✅ 관문 판정 불변 확인 — 이 셀은 해석만 더한다")
    run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 그림 + 마무리
import matplotlib.pyplot as plt
R = CONFIG.get("result", {})
if R.get("stopped_at") == "test" and R.get("test_S"):
    D = R["test_S"]
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    o = np.argsort(-np.array(D["se"]))
    ax[0].errorbar(range(len(o)), np.array(D["auroc"])[o], yerr=1.96 * np.array(D["se"])[o],
                   fmt="o", ms=4, capsize=2)
    ax[0].axhline(D["macro"], color="C1", ls="--", label=f"매크로 {D['macro']:.4f}")
    ax[0].axhspan(D["lo"], D["hi"], color="C1", alpha=.15, label="레코드 부트 95% CI")
    ax[0].set_title(f"① TEST 레코드별 S AUROC (GMIN={R['gmin']} · SE 큰 순)")
    ax[0].set_xlabel("레코드"); ax[0].set_ylabel("AUROC"); ax[0].legend(); ax[0].grid(alpha=.3)
    rows = [r for r in CONFIG["dev_sweep"] if r.get("n")]
    ax[1].plot([r["gmin"] for r in rows], [r["se_max"] for r in rows], "o-", label="DEV 최대 SE")
    ax[1].plot([r["gmin"] for r in rows], [r["se_med"] for r in rows], "s-", label="DEV 중앙 SE")
    ax[1].axhline(SE_MAX, color="crimson", ls="--", label=f"최대 상한 {SE_MAX}")
    ax[1].axhline(SE_CAP, color="gray", ls=":", label=f"중앙 상한 {SE_CAP}")
    ax[1].axvline(R["gmin"], color="C2", ls="-.", label=f"선택 GMIN={R['gmin']}")
    ax[1].set_xscale("log"); ax[1].set_xlabel("GMIN_S"); ax[1].set_ylabel("부트 SE")
    ax[1].set_title("② DEV 스윕 — 여기서만 골랐다"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    plt.tight_layout(); run.save_fig("q7b_svdb_se", fig); plt.show()

run.finish(CONFIG.get("result", {"verdicts": {}, "stopped_at": "unknown"}))
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-se-check`")
